# Huấn luyện YOLOv8 trên Google Colab

Notebook này được chỉnh để bạn có thể chạy trực tiếp trên Colab với dataset nằm trên Google Drive. Các bước bên dưới đi từ upload dữ liệu, kiểm tra path, tạo file YAML cho Colab, đến train và xem kết quả.

## Bước 0: Chuẩn bị dữ liệu trên Google Drive

Cách dễ nhất là upload cả thư mục dự án `yolov8_custom_finetune` lên `MyDrive` để giữ nguyên cấu trúc.

Cấu trúc nên có dạng:

```text
MyDrive/
  yolov8_custom_finetune/
    configs/
    scripts/
    requirements.txt
    datasets/
      leanbot_yolo/
        classes.txt
        images/
          train/
          val/
        labels/
          train/
          val/
```

Nếu tên thư mục trên Drive khác với `yolov8_custom_finetune`, chỉ cần sửa biến `PROJECT_DIR` ở cell tiếp theo.

Trên Colab, đường dẫn Drive thường bắt đầu bằng:

```python
/content/drive/MyDrive/...
```

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Sửa lại nếu tên thư mục dự án trên Drive của bạn khác.
PROJECT_DIR = Path('/content/drive/MyDrive/yolov8_custom_finetune')
DATASET_DIR = PROJECT_DIR / 'datasets' / 'leanbot_yolo'

print('PROJECT_DIR =', PROJECT_DIR)
print('DATASET_DIR =', DATASET_DIR)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'Khong tim thay PROJECT_DIR: {PROJECT_DIR}\n'
        'Hay kiem tra lai ten thu muc du an tren Google Drive.'
    )

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f'Khong tim thay DATASET_DIR: {DATASET_DIR}\n'
        'Hay kiem tra lai duong dan dataset tren Google Drive.'
    )

## Bước 1: Kiểm tra path và số lượng dữ liệu

Cell này giúp bạn xác nhận Colab đang nhìn thấy đúng dataset trên Drive. Nếu kết quả đếm ảnh hoặc label bằng `0`, thường là path đang sai hoặc upload thiếu thư mục con.

In [ ]:
from pathlib import Path

for path in [
    DATASET_DIR,
    DATASET_DIR / 'images' / 'train',
    DATASET_DIR / 'images' / 'val',
    DATASET_DIR / 'labels' / 'train',
    DATASET_DIR / 'labels' / 'val',
]:
    print(path, '->', path.exists())

train_images = len(list((DATASET_DIR / 'images' / 'train').glob('*')))
val_images = len(list((DATASET_DIR / 'images' / 'val').glob('*')))
train_labels = len(list((DATASET_DIR / 'labels' / 'train').glob('*.txt')))
val_labels = len(list((DATASET_DIR / 'labels' / 'val').glob('*.txt')))

print('train_images =', train_images)
print('val_images =', val_images)
print('train_labels =', train_labels)
print('val_labels =', val_labels)

print('\nVi du mot vai file trong dataset:')
for sample in sorted(DATASET_DIR.rglob('*'))[:12]:
    print(sample)

## Bước 2: Cài thư viện và kiểm tra môi trường Colab

Nếu bạn dùng Colab GPU, nên vào `Runtime > Change runtime type > T4 GPU` trước khi chạy cell này.

In [ ]:
import os
from pathlib import Path

os.chdir(PROJECT_DIR)
print('Current working directory:', os.getcwd())

candidate_requirements = [
    PROJECT_DIR / 'requirements.txt',
    PROJECT_DIR / 'requirement.txt',
]
requirements_path = next((path for path in candidate_requirements if path.exists()), None)
print('requirements_path =', requirements_path)
print('requirements_exists =', requirements_path is not None)

print('\nNoi dung thu muc du an:')
for item in sorted(PROJECT_DIR.iterdir())[:20]:
    print('-', item.name)

if requirements_path is not None:
    print(f'\nDang cai thu vien tu {requirements_path}')
    !pip install -q -r "$requirements_path"
else:
    print('\nKhong tim thay requirements.txt, dang cai bo thu vien toi thieu truc tiep.')
    !pip install -q ultralytics opencv-python pandas matplotlib

import ultralytics
import torch

ultralytics.checks()
print('torch.cuda.is_available() =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))

## Bước 3: Tạo file YAML dành riêng cho Colab

Thay vì sửa tay `configs/data.yaml`, cell này sẽ tự tạo file `configs/leanbot_data_colab.yaml` với đường dẫn tuyệt đối trên Drive. Nếu `classes.txt` tồn tại, tên class sẽ được lấy tự động từ đó.

In [ ]:
classes_path = DATASET_DIR / 'classes.txt'
class_names = [line.strip() for line in classes_path.read_text().splitlines() if line.strip()] if classes_path.exists() else ['Leanbot']

names_block = '\n'.join(f'  {idx}: {name}' for idx, name in enumerate(class_names))
yaml_text = f'''path: {DATASET_DIR}
train: images/train
val: images/val

names:
{names_block}
'''

yaml_path = PROJECT_DIR / 'configs' / 'leanbot_data_colab.yaml'
yaml_path.write_text(yaml_text)

print('Da tao:', yaml_path)
print(yaml_path.read_text())

## Bước 4: Train trên Colab

Script `scripts/train.py` trong dự án đã có augmentation xoay và lật ảnh ngẫu nhiên ở mỗi epoch.

Gợi ý:
- Nếu dùng GPU Colab, giữ `--device 0`.
- Nếu báo thiếu RAM GPU, giảm `--batch` từ `16` xuống `8` hoặc `4`.
- Nếu bạn chỉ muốn chạy thử nhanh, đổi `--epochs 50` thành `20`.

In [ ]:
import os

os.chdir(PROJECT_DIR)

!python scripts/train.py \
  --data configs/leanbot_data_colab.yaml \
  --epochs 50 \
  --batch 16 \
  --device 0 \
  --name leanbot_colab

## Bước 5: Xem kết quả sau khi train

Trọng số tốt nhất thường nằm tại:

```text
runs/detect/leanbot_colab/weights/best.pt
```

Vì dự án đang nằm trên Drive, file này cũng sẽ được lưu luôn trên Google Drive.

In [ ]:
from IPython.display import Image, display

run_dir = PROJECT_DIR / 'runs' / 'detect' / 'leanbot_colab'
weights_dir = run_dir / 'weights'

print('Run dir   :', run_dir)
print('Best model:', weights_dir / 'best.pt')
print('Last model:', weights_dir / 'last.pt')

for file_name in ['labels.jpg', 'results.png', 'PR_curve.png', 'F1_curve.png', 'confusion_matrix_normalized.png']:
    file_path = run_dir / file_name
    if file_path.exists():
        print('\nHien thi:', file_path)
        display(Image(filename=str(file_path)))

## Ghi chú nhanh

- Nếu bạn chưa chắc thư mục nào đang nằm trong `MyDrive`, có thể chạy `!find /content/drive/MyDrive -maxdepth 3 -type d | sort` để dò path.
- Nếu không dùng GPU, đổi `--device 0` thành `--device cpu`, nhưng train sẽ chậm hơn nhiều.
- Sau khi có `best.pt`, bạn có thể tải về máy hoặc dùng tiếp cho suy luận trên ảnh, video, webcam.